# 실습 3. 치료 전과 후의 크기 비교

사회약학 연구방법 노트의 '실습 3. 치료 전과 후의 크기 비교'에 딸린 노트북입니다. Student(1908)가 t 검정의 예제로 쓴 sleep 자료(같은 환자 10명이 두 수면제를 모두 복용)로 대응표본 t 검정과 Wilcoxon 부호순위 검정을 합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 3에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 sleep 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import scipy, matplotlib

print("pandas    ", pd.__version__)
print("numpy     ", np.__version__)
print("scipy     ", scipy.__version__)
print("matplotlib", matplotlib.__version__)

### 셀 2. 데이터 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/datasets/sleep.csv")
sleep = pd.read_csv(url)
sleep

### 셀 3. 짝 확인하기

In [ ]:
pd.crosstab(sleep["ID"], sleep["group"])

### 셀 4. 넓은 형식으로 바꾸기

In [ ]:
wide = sleep.pivot(index="ID", columns="group", values="extra")
wide.columns = ["drug1", "drug2"]
wide["d"] = wide["drug2"] - wide["drug1"]   # 환자별 차이
wide

pivot에 같은 짝이 두 번 있으면 나는 오류를 일부러 내 보는 셀(셀 5)은 이 노트북에서 뺐습니다(사이트의 실습 3 a 절에 있습니다).

### 셀 6. 요약통계와 상관

In [ ]:
r = wide["drug1"].corr(wide["drug2"])
print(f"두 약의 상관계수 r = {r:.3f}")
wide.describe().round(2)

### 셀 7. 짝지은 그림

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.6))
for pid, row in wide.iterrows():            # 환자 한 명씩
    ax1.plot([1, 2], [row["drug1"], row["drug2"]],
             marker="o", color="gray")
ax1.set_xticks([1, 2], ["Drug 1", "Drug 2"])
ax1.set_xlim(0.7, 2.3)
ax1.set_ylabel("Extra sleep (hours)")

ax2.scatter(wide.index, wide["d"], color="black")
ax2.axhline(0, ls="--", color="gray")
ax2.set_xticks(wide.index)
ax2.set_xlabel("Patient ID")
ax2.set_ylabel("Drug 2 - Drug 1 (hours)")
plt.tight_layout()
plt.show()

## b. 대응표본 t 검정

### 셀 8. 대응표본 t 검정

In [ ]:
res = stats.ttest_rel(wide["drug2"], wide["drug1"])
print(res)
print(f"t = {res.statistic:.3f}, p = {res.pvalue:.4f}, "
      f"df = {res.df}")
ci = res.confidence_interval()
print(f"mean d = {wide['d'].mean():.2f} h, "
      f"95% CI {ci.low:.2f} ~ {ci.high:.2f}")

### 셀 9. 차이 d에 대한 일표본 t 검정

In [ ]:
d = wide["d"]
one = stats.ttest_1samp(d, popmean=0)
print(f"one-sample: t = {one.statistic:.3f}, p = {one.pvalue:.4f}")

se = d.std() / np.sqrt(len(d))       # 차이의 표준오차
print(f"mean = {d.mean():.2f}, SD = {d.std():.3f}, "
      f"SE = {se:.3f}, t = {d.mean() / se:.3f}")

### 셀 10. 잘못된 분석: 독립표본 t 검정

In [ ]:
wrong = stats.ttest_ind(wide["drug2"], wide["drug1"])  # 잘못된 분석
ci_w = wrong.confidence_interval()
print(f"t = {wrong.statistic:.3f}, p = {wrong.pvalue:.4f}, "
      f"df = {wrong.df:.0f}")
print(f"95% CI {ci_w.low:.2f} ~ {ci_w.high:.2f}")

### 셀 11. 상관이 표준오차를 줄이는 원리

In [ ]:
s1, s2 = wide["drug1"].std(), wide["drug2"].std()
sd_paired = np.sqrt(s1**2 + s2**2 - 2 * r * s1 * s2)
sd_indep = np.sqrt(s1**2 + s2**2)            # r = 0으로 본 경우
print(f"SD1 = {s1:.3f}, SD2 = {s2:.3f}, r = {r:.3f}")
print(f"SD of d: paired {sd_paired:.3f}, "
      f"if r = 0 {sd_indep:.3f}")

### 셀 12. 차이 d의 정규성

In [ ]:
sw = stats.shapiro(d)
print(f"Shapiro-Wilk W = {sw.statistic:.3f}, p = {sw.pvalue:.3f}")

fig, ax = plt.subplots(figsize=(4.2, 3.6))
stats.probplot(d, dist="norm", plot=ax)     # 차이 d의 정규 Q-Q plot
ax.set_title("Normal Q-Q plot of d")
plt.show()

## c. Wilcoxon 부호순위 검정

### 셀 13. Wilcoxon 부호순위 검정

In [ ]:
w = stats.wilcoxon(d)
print(w)

### 셀 14. 부호순위를 직접 계산하기

In [ ]:
nz = d[d != 0]                   # 차이가 0인 쌍은 제외
rk = stats.rankdata(nz.abs())    # |d|에 순위 (동점은 평균순위)
print(pd.DataFrame({"d": nz, "rank": rk}).T)
w_plus = rk[nz > 0].sum()        # 늘어난 쪽의 순위합
w_minus = rk[nz < 0].sum()       # 줄어든 쪽의 순위합
print("n =", len(nz), " W+ =", w_plus, " W- =", w_minus)

### 셀 15. p-value 계산 방식 비교

In [ ]:
g = stats.wilcoxon(d, alternative="greater")
print(f"greater: statistic = {g.statistic}, p = {g.pvalue:.5f}")
a = stats.wilcoxon(d, method="asymptotic")
print(f"asymptotic: z = {a.zstatistic:.3f}, p = {a.pvalue:.4f}")
ac = stats.wilcoxon(d, method="asymptotic", correction=True)
print(f"asymptotic + correction: p = {ac.pvalue:.4f}")
pr = stats.wilcoxon(d, zero_method="pratt", method="asymptotic")
print(f"pratt (asymptotic): p = {pr.pvalue:.4f}")

### 셀 16. 부호검정

In [ ]:
n_pos = int((d > 0).sum())      # drug2에서 더 잔 환자
n_neg = int((d < 0).sum())      # drug1에서 더 잔 환자
print("+:", n_pos, " -:", n_neg, " 0:", int((d == 0).sum()))
print(stats.binomtest(n_pos, n_pos + n_neg, p=0.5))

### 셀 17. 중앙값과 사분위수

In [ ]:
print(wide.median())
print(d.quantile([0.25, 0.5, 0.75]))

## 과제 정답

**과제 1.** 셀 8의 출력 가운데 `mean d = 1.58 h, 95% CI 0.70 ~ 2.46` 행을 p-value와 함께 논문의 Results처럼 한 문장으로 옮기고, 이 결과가 말해 주지 않는 것을 두 가지 이상 쓰세요. 환자별 차이와 각 약의 평균을 확인해 근거로 쓰세요.

### 셀 18. 과제 1 정답. 대응표본 t 검정의 한 행 읽기

In [ ]:
print(d.round(1).sort_values().tolist())       # 환자별 차이, 작은 순서로
print("median d:", d.median())
print("d >= 1 h:", int((d >= 1).sum()), "of", len(d))
wide[["drug1", "drug2"]].mean().round(2).rename("mean")   # 약을 쓰지 않은 밤 대비

**과제 2.** 환자 9의 차이 d가 4.6이 아니라 14.6이었다고 하고(약 2의 효과가 더 커 보이는 방향), 대응표본 t 검정, Wilcoxon 부호순위 검정, 부호검정을 원래 자료와 나란히 다시 돌리세요. 어느 검정의 P가 바뀌고 어느 것이 그대로인지, 왜 그런지 설명하세요.

### 셀 19. 과제 2 정답. 극단값 하나가 커지면 세 검정은

In [ ]:
def three_tests(x):                    # 차이 x에 세 검정을 한 번에
    t = stats.ttest_1samp(x, 0)
    return {"mean": x.mean(), "SD": x.std(),
            "paired t p": t.pvalue,
            "signed-rank p": stats.wilcoxon(x).pvalue,
            "sign test p": stats.binomtest(int((x > 0).sum()),
                                           int((x != 0).sum())).pvalue}

d2 = d.copy()
d2[9] = 14.6                           # 환자 9의 차이만 4.6 -> 14.6
pd.DataFrame({"d": three_tests(d), "d2": three_tests(d2)}).round(4)

**과제 3.** 동료가 sleep 자료를 값 크기순으로 정렬한 뒤 약 1과 약 2의 값을 각각 꺼내 정규성을 확인하고 `stats.ttest_rel`에 넣었습니다. 그리고 "두 약의 값이 각각 정규분포를 따랐으므로(Shapiro-Wilk P = 0.41, 0.35) 대응표본 t 검정을 했다. 약 2는 약 1보다 수면을 평균 1.58시간 더 늘렸다(95% CI 1.19–1.97, t(9) = 9.27, P < 0.001)"라고 썼습니다. 동료의 분석을 재현하고, 틀린 곳 두 가지를 찾아 왜 틀렸는지 설명하세요.

### 셀 20. 과제 3 정답. 정렬한 뒤 행 순서로 짝지은 분석

In [ ]:
s = sleep.sort_values("extra")                  # 동료의 코드: 값 크기순 정렬
drug1_s = s.loc[s["group"] == 1, ["ID", "extra"]]
drug2_s = s.loc[s["group"] == 2, ["ID", "extra"]]
print("Shapiro p (drug 1, drug 2):",
      round(stats.shapiro(drug1_s["extra"]).pvalue, 2),
      round(stats.shapiro(drug2_s["extra"]).pvalue, 2))
res_c = stats.ttest_rel(drug2_s["extra"], drug1_s["extra"])
ci_c = res_c.confidence_interval()
print(f"colleague: t = {res_c.statistic:.2f}, p = {res_c.pvalue:.1e}")
print(f"           95% CI {ci_c.low:.2f} ~ {ci_c.high:.2f}")
pd.DataFrame({"ID (drug 1)": drug1_s["ID"].to_numpy(),
              "ID (drug 2)": drug2_s["ID"].to_numpy(),
              "d used": (drug2_s["extra"].to_numpy()
                         - drug1_s["extra"].to_numpy()).round(1)})